phase1


In [ ]:
#GSE70138_mapping:
import h5py
import pandas as pd

file_path = "/sybig/home/shs/projects/project1_L1000/datasets/lincs/NCBI/GSE70138_Level3_2017.gctx"
with h5py.File(file_path, 'r') as f:
    data = f['0/DATA/0/matrix'][:] 
    row_names = [r.decode('utf-8') for r in f['0/META/COL/id'][:]]  
    col_names = [c.decode('utf-8') for c in f['0/META/ROW/id'][:]]  

df = pd.DataFrame(data, index=row_names, columns=col_names)

print("Shape (samples x genes):", df.shape)
print("First 5 rows, first 10 columns:")
print(df.iloc[:5, :10])
print("First 10 column names (genes):", df.columns[:10])
print("First 10 sample IDs:", df.index[:10])


In [ ]:
#GSE92742_mapping:
import h5py
import pandas as pd

file_path = "/sybig/home/shs/projects/project1_L1000/datasets/lincs/NCBI/GSE92742_level3.gctx"
with h5py.File(file_path, 'r') as f:
    data = f['0/DATA/0/matrix'][:] 
    row_names = [r.decode('utf-8') for r in f['0/META/COL/id'][:]]  
    col_names = [c.decode('utf-8') for c in f['0/META/ROW/id'][:]]  

df = pd.DataFrame(data, index=row_names, columns=col_names)

print("Shape (samples x genes):", df.shape)
print("First 5 rows, first 10 columns:")
print(df.iloc[:5, :10])
print("First 10 column names (genes):", df.columns[:10])
print("First 10 sample IDs:", df.index[:10])


In [ ]:
#number_unique_cell
import h5py
import h5py
import pandas as pd

file_path = ""
with h5py.File(file_path, 'r') as f:
    data = f['0/DATA/0/matrix'][:] 
    row_names = [r.decode('utf-8') for r in f['0/META/COL/id'][:]]  
    col_names = [c.decode('utf-8') for c in f['0/META/ROW/id'][:]]  

df = pd.DataFrame(data, index=row_names, columns=col_names)

cell_lines = [name.split('_')[1] for name in df.index]
unique_cell_lines = set(cell_lines)
print("\nNumber of unique cell lines:", len(unique_cell_lines))
print("Cell lines:", unique_cell_lines)

In [ ]:
#frequency_cell_line
import h5py
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

file_path = ""

with h5py.File(file_path, 'r') as f:
    data = f['0/DATA/0/matrix'][:] 
    row_names = [r.decode('utf-8') for r in f['0/META/COL/id'][:]]  
    col_names = [c.decode('utf-8') for c in f['0/META/ROW/id'][:]]  

df = pd.DataFrame(data, index=row_names, columns=col_names)

# Extract cell lines (still using string splitting for now)
cell_lines = [name.split('_')[1] for name in df.index] 

# Count frequency per cell line
cell_counts = pd.Series(cell_lines).value_counts()
cell_counts = cell_counts.sort_values(ascending=True)

# Scale to 10^3
frequencies_scaled = cell_counts.values / 1e3

plt.figure(figsize=(10, 8))
plt.barh(cell_counts.index, frequencies_scaled, color='purple')
plt.xlabel('Frequency (×10³)')
plt.ylabel('Cell line')
plt.title('Frequency per cell line - GSE70138')
plt.show()


In [ ]:
#Metadata_colums
import os
import pandas as pd

meta_folder = "/sybig/home/shs/projects/project1_L1000/datasets/lincs/NCBI/Mtext/GSE92742_metadata"

cell_info       = pd.read_csv(os.path.join(meta_folder, "GSE92742_Broad_LINCS_cell_info.txt"), sep='\t')
gene_info       = pd.read_csv(os.path.join(meta_folder, "GSE92742_Broad_LINCS_gene_info.txt"), sep='\t')
pert_info       = pd.read_csv(os.path.join(meta_folder, "GSE92742_Broad_LINCS_pert_info.txt"), sep='\t')
pert_metrics    = pd.read_csv(os.path.join(meta_folder, "GSE92742_Broad_LINCS_pert_metrics.txt"), sep='\t')
sig_info        = pd.read_csv(os.path.join(meta_folder, "GSE92742_Broad_LINCS_sig_info.txt"), sep='\t')
sig_metrics     = pd.read_csv(os.path.join(meta_folder, "GSE92742_Broad_LINCS_sig_metrics.txt"), sep='\t')
inst_info       = pd.read_csv(os.path.join(meta_folder, "GSE92742_Broad_LINCS_inst_info.txt"), sep='\t')

print("Cell info columns:", cell_info.columns.tolist())
print("Gene info columns:", gene_info.columns.tolist())
print("Pert info columns:", pert_info.columns.tolist())
print("Pert metrics columns:", pert_metrics.columns.tolist())
print("Signature info columns:", sig_info.columns.tolist())
print("Signature metrics columns:", sig_metrics.columns.tolist())
print("inst_info columns:", inst_info.columns.tolist())



In [ ]:
#merging_level3_with_sig_info
import pandas as pd
from cmapPy.pandasGEXpress.parse import parse
import numpy as np

# Load the metadata file for GSE92742
sig_info_path = "/sybig/home/shs/projects/project1_L1000/datasets/lincs/NCBI/Mtext/GSE92742_metadata/GSE92742_Broad_LINCS_sig_info.txt"
sig_info = pd.read_csv(sig_info_path, sep="\t", low_memory=False)

# Select the relevant metadata columns based on the available columns
sig_info_small = sig_info[[
    "sig_id", "pert_id", "pert_iname", "pert_type", 
    "cell_id", "pert_idose", "pert_itime", "distil_id"
]].copy()

# Exploding `distil_id` to have one row per replicate
sig_info_small["distil_id"] = sig_info_small["distil_id"].str.split("|")
sig_info_expanded = sig_info_small.explode("distil_id").reset_index(drop=True)

# Load the GCTX file for GSE92742 (expression matrix)
gctx_path = "/sybig/home/shs/projects/project1_L1000/datasets/lincs/NCBI/GSE92742_level3.gctx"

# Load only column metadata (sample IDs)
from cmapPy.pandasGEXpress.parse import parse
col_meta = parse(gctx_path, col_meta_only=True)
gctx_ids = col_meta.index.tolist()

# Find the common distil_ids between the metadata and the expression matrix
valid_ids = list(set(sig_info_expanded["distil_id"]).intersection(set(gctx_ids)))

# Filter the metadata to keep only matching distil_ids
sig_info_expanded_filtered = sig_info_expanded[sig_info_expanded["distil_id"].isin(valid_ids)].copy()

# Load expression data for the matching distil_ids
gctoo = parse(gctx_path, cid=valid_ids)

# Convert the expression data to float32 for memory efficiency
expr = gctoo.data_df.astype("float32").T  # Transpose: rows = distil_id, columns = genes

# Check the filtered metadata and expression matrix shape
print("Filtered metadata shape:", sig_info_expanded_filtered.shape)
print("Expression matrix shape (samples x genes):", expr.shape)

# Ensure `distil_id` is the index in the metadata
sig_info_expanded_filtered.set_index("distil_id", inplace=True)

# Filter the expression data to only include the matching `distil_id`s
expr_filtered = expr.loc[expr.index.isin(sig_info_expanded_filtered.index)]

# Merge the filtered metadata with the expression data
merged_data = sig_info_expanded_filtered.loc[expr_filtered.index].join(expr_filtered)

# Reset the index to get `distil_id` back as a column
merged_data.reset_index(inplace=True)

# Check the merged dataset
print("Merged dataset shape:", merged_data.shape)
print("Mapped dataset columns:", merged_data.columns.tolist())
print(merged_data.head())




In [ ]:
#keep_landmarks_gene
import pandas as pd

# Path to the gene info file
gene_metadata_path = '/sybig/home/shs/projects/project1_L1000/datasets/lincs/NCBI/Mtext/GSE92742_metadata/GSE92742_Broad_LINCS_gene_info.txt'

# Load the gene metadata (assuming it's tab-separated)
gene_metadata = pd.read_csv(gene_metadata_path, sep='\t')

# Inspect the first few rows to understand the structure
print(gene_metadata.head())

# Step 2: Filter for landmark genes
landmark_gene_metadata = gene_metadata[gene_metadata['pr_is_lm'] == 1]

# Check the first few rows of the filtered data
print(landmark_gene_metadata.head())

# Step 3: Extract the list of landmark gene IDs
landmark_gene_ids = landmark_gene_metadata['pr_gene_id'].tolist()

# Check the first few gene IDs to verify
print(landmark_gene_ids[:10])  # Display the first 10 gene IDs

# Step 4: Filter the merged dataset to keep only landmark genes
merged_landmark_data = merged_data[['cid', 'sig_id', 'pert_id', 'pert_iname', 'pert_type', 'cell_id', 'pert_idose', 'pert_itime'] + [str(gene_id) for gene_id in landmark_gene_ids]]

# Check the filtered dataset
print("Filtered dataset shape:", merged_landmark_data.shape)
print("Filtered dataset columns:", merged_landmark_data.columns.tolist())
print(merged_landmark_data.head())


In [ ]:
# Step: Apply filter for 'trt_cp' and 'ctl_vehicle' on the merged_landmark_data
merged_landmark_filtered = merged_landmark_data[merged_landmark_data['pert_type'].isin(['trt_cp', 'ctl_vehicle'])]

# Check the filtered dataset
print("Filtered dataset shape:", merged_landmark_filtered.shape)
print("Filtered dataset columns:", merged_landmark_filtered.columns.tolist())
print(merged_landmark_filtered.head())


In [ ]:
merged_landmark_filtered['dataset_name'] = 'dataset_name'

# Check the updated dataset
print("Updated dataset with dataset name column:")
print(merged_landmark_filtered.head())

In [ ]:
#adding_conanical_smiles
import pandas as pd
import os

# Define the path to the metadata folder
meta_folder = "/sybig/home/shs/projects/project1_L1000/datasets/lincs/NCBI/Mtext/GSE92742_metadata"

# Load the pert_info file (tab-separated)
pert_info = pd.read_csv(os.path.join(meta_folder, "GSE92742_Broad_LINCS_pert_info.txt"), sep='\t')

# Check the first few rows to confirm the structure and columns
print(pert_info.head())

# Merge pert_info with filtered_data based on 'pert_id'
filtered_data = filtered_data.merge(pert_info[['pert_id', 'canonical_smiles']], 
                                   on='pert_id', 
                                   how='left')  # 'left' ensures all rows in filtered_data are kept

# Verify the merge and check a sample of the data
print(filtered_data[['pert_id', 'canonical_smiles']].sample(20))

# Filter out rows where 'canonical_smiles' is invalid
filtered_data = filtered_data[~filtered_data['canonical_smiles'].isin(['-666', 'restricted', 'nan', np.nan])]

# Check the shape again after filtering
print(f"Shape after filtering invalid rows: {filtered_data.shape}")

In [ ]:
#checking validiti of smiles
import numpy as np
from rdkit import Chem

# Check if RDKit is available
try:
    from rdkit import Chem
    RDKIT_AVAILABLE = True
except ImportError:
    RDKIT_AVAILABLE = False

# SMILES validation function
def is_valid_smiles(smiles):
    if smiles in ['-666', 'restricted', 'nan', None, np.nan]:
        return False  # Invalid SMILES

    # Attempt to parse the SMILES string
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return False  # If molecule is None, it's an invalid SMILES
    
    # Optionally, sanitize the molecule to check its validity
    try:
        Chem.SanitizeMol(mol)
        return True
    except:
        return False

# If RDKit is available, validate and filter the SMILES
if RDKIT_AVAILABLE:
    filtered_data['valid_smiles'] = filtered_data['canonical_smiles'].apply(is_valid_smiles)

    # Filter out rows with invalid SMILES
    filtered_data = filtered_data[filtered_data['valid_smiles']]

    # Check the results
    print(f"Rows remaining after filtering invalid SMILES: {len(filtered_data)}")

    # Display a sample of the cleaned data
    print(filtered_data[['pert_id', 'canonical_smiles', 'valid_smiles']].sample(20))

In [ ]:
import anndata
import re
import pandas as pd

# Load the dataset (combine2.h5ad)
adata = anndata.read_h5ad('/sybig/home/shs/projects/project1_L1000/datasets/lincs/merge/combine2.h5ad')

# Check if adata.obs has a MultiIndex
if isinstance(adata.obs.index, pd.MultiIndex):
    print("MultiIndex detected. Flattening the index...")
    # Flatten the MultiIndex to a single index by resetting it
    adata.obs = adata.obs.reset_index(drop=True)

# Function to extract dose and unit from pert_idose, while handling -666 cases
def split_dose_and_unit(pert_idose):
    # Handle -666 cases
    if pert_idose == '-666':
        return '-666', '-666'  # Assign -666 for both dose and unit
    
    # Regular expression to extract numeric dose and unit
    match = re.match(r"([0-9\.]+)\s*(\D+)", pert_idose)  # Match numeric and unit parts
    if match:
        dose = float(match.group(1))  # Numeric dose
        unit = match.group(2).strip()  # Unit of measure
        return dose, unit
    else:
        return None, None  # If the format does not match

# Initialize empty lists for dose and unit
pert_doses = []
pert_dose_units = []

# Iterate through each entry in the 'pert_idose' column and apply the split function
for dose in adata.obs['pert_idose']:
    dose_val, unit_val = split_dose_and_unit(dose)
    pert_doses.append(dose_val)
    pert_dose_units.append(unit_val)

# Assign the lists as new columns in the DataFrame
adata.obs['pert_dose'] = pert_doses
adata.obs['pert_dose_unit'] = pert_dose_units

# Display the first few rows of the resulting DataFrame
print(adata.obs[['pert_idose', 'pert_dose', 'pert_dose_unit']].head())

In [ ]:
import pandas as pd

# Function to convert all doses to micromolar (µM)
def convert_to_micromolar(row):
    dose = row['pert_dose']
    unit = row['pert_dose_unit']
    
    if unit in ['um', 'µM']:  # Already micromolar
        return dose, 'µM'
    elif unit == 'nM':         # Nanomolar to micromolar
        return dose / 1000, 'µM'
    else:                      # Unrecognized units, leave as is
        return dose, unit

# Apply conversion
adata3.obs[['pert_dose', 'pert_dose_unit']] = adata3.obs.apply(
    convert_to_micromolar, axis=1, result_type='expand'
)

# Verify
print(adata3.obs[['pert_dose', 'pert_dose_unit']].head())

In [ ]:
# Filter out rows where 'pert_dose' is 0.0
adata2 = adata2[adata2.obs['pert_dose'] != 0.0]

# Verify by checking the shape of the resulting dataset
print(f"Shape after deleting samples with 'pert_dose' == 0.0: {adata2.shape}")

chembel

In [ ]:
import pandas as pd

# File paths
file1 = "/sybig/home/shs/projects/project1_L1000/datasets/lincs/NCBI/Mtext/GSE92742_metadata/GSE92742_Broad_LINCS_pert_info.txt"
file2 = "/sybig/home/shs/projects/project1_L1000/datasets/lincs/NCBI/Mtext/GSE70138_metadata/GSE70138_Broad_LINCS_pert_info_2017-03-06.txt"

# Columns to keep
columns_to_keep = ['pert_id', 'canonical_smiles', 'inchi_key', 'pert_iname', 'pert_type']

# Read files
df1 = pd.read_csv(file1, sep="\t", usecols=columns_to_keep)
df2 = pd.read_csv(file2, sep="\t", usecols=columns_to_keep)

# Add dataset column
df1['dataset'] = 'GSE92742'
df2['dataset'] = 'GSE70138'

# Combine
combined_df = pd.concat([df1, df2], ignore_index=True)

# Save combined file
combined_df.to_csv("/sybig/home/shs/projects/project1_L1000/datasets/lincs/combined_pert_info.txt",
                   sep="\t", index=False)

print("Combined dataframe shape:", combined_df.shape)
print("Columns:", combined_df.columns.tolist())

In [ ]:
filtered_df = combined_df[combined_df['pert_type'].isin(['trt_cp', 'ctl_vehicle'])]
print("Filtered dataframe shape:", filtered_df.shape)
# Save filtered file
filtered_df.to_csv("/sybig/home/shs/projects/project1_L1000/datasets/lincs/combined_pert_info_filtered.txt",
                   sep="\t", index=False)

In [ ]:
import scanpy as sc
import pandas as pd

# Paths
adata_path = "/sybig/home/shs/projects/project1_L1000/datasets/lincs/merge/Level3_twophase.h5ad"
pert_info_path = "/sybig/home/shs/projects/project1_L1000/datasets/lincs/combined_pert_info_filtered.txt"

# Load AnnData
adata = sc.read_h5ad(adata_path)

# Load combined pert_info
pert_info = pd.read_csv(pert_info_path, sep="\t")

# Keep only needed columns and remove duplicates
pert_info = pert_info[['canonical_smiles', 'inchi_key']].drop_duplicates()

# Merge with adata.obs (assuming canonical_smiles exists there)
adata.obs = adata.obs.merge(
    pert_info,
    on="canonical_smiles",
    how="left"
)

# Save updated AnnData
adata.write("/sybig/home/shs/projects/project1_L1000/datasets/lincs/merge/Level3_twophase_with_inchi.h5ad")

print("Finished. New obs columns:")
print(adata.obs.columns)

In [ ]:
import scanpy as sc

# Load the updated AnnData
adata_path = "/sybig/home/shs/projects/project1_L1000/datasets/lincs/merge/Level3_twophase_with_inchi.h5ad"
adata = sc.read_h5ad(adata_path)

# 1️⃣ Check basic info
print("AnnData shape (cells x features):", adata.shape)
print("Observation columns:", adata.obs.columns.tolist())

# 2️⃣ Check that inchi_key was added
if 'inchi_key' in adata.obs.columns:
    print("inchi_key column exists.")
else:
    print("ERROR: inchi_key column missing!")

# 3️⃣ Check for missing inchi_keys
num_missing = adata.obs['inchi_key'].isna().sum()
total = adata.obs.shape[0]
print(f"Missing inchi_keys: {num_missing} / {total} ({num_missing/total*100:.2f}%)")

# 4️⃣ Check for duplicates in obs if needed
num_duplicates = adata.obs['canonical_smiles'].duplicated().sum()
print(f"Duplicate canonical_smiles in obs: {num_duplicates}")

# 5️⃣ Quick peek at merged data
print(adata.obs[['canonical_smiles', 'inchi_key']].head(10))

In [ ]:
import pandas as pd

# Path to the ChEMBL chemical representations file
chembl_file = "/sybig/home/shs/projects/project1_L1000/datasets/Chembel/chembl_36_chemreps.txt"

# Load the file
chembl_reps = pd.read_csv(chembl_file, sep="\t")

# Inspect the first few rows
print(chembl_reps.head())

# Check columns
print("Columns:", chembl_reps.columns.tolist())

# Check shape
print("Shape:", chembl_reps.shape)

In [ ]:
import scanpy as sc
import pandas as pd

# Paths
adata_path = "/sybig/home/shs/projects/project1_L1000/datasets/lincs/merge/Level3_twophase_with_inchi.h5ad"
chembl_file = "/sybig/home/shs/projects/project1_L1000/datasets/Chembel/chembl_36_chemreps.txt"

# Load AnnData
adata = sc.read_h5ad(adata_path)

# Load ChEMBL data
chembl_df = pd.read_csv(chembl_file, sep="\t")  # adjust sep if needed
print("ChEMBL columns:", chembl_df.columns.tolist())

# Keep only relevant columns
chembl_df = chembl_df[['chembl_id', 'standard_inchi_key']].drop_duplicates()

# Merge with adata.obs using inchi_key
adata.obs = adata.obs.merge(
    chembl_df,
    left_on='inchi_key',
    right_on='standard_inchi_key',
    how='left'
)

# Drop the extra column if you want
adata.obs.drop(columns=['standard_inchi_key'], inplace=True)


# Quick check
print("AnnData obs columns after merge:", adata.obs.columns.tolist())
print("Number of rows with ChEMBL ID:", adata.obs['chembl_id'].notna().sum())

In [ ]:
import scanpy as sc
import pandas as pd
import sqlite3

# Paths
adata_path = "/sybig/home/shs/projects/project1_L1000/datasets/lincs/merge/Level3_twophase_with_chembl.h5ad"
chembl_db = "/sybig/home/shs/projects/project1_L1000/datasets/Chembel/chembl_36/chembl_36_sqlite/chembl_36.db"

# Load AnnData
adata = sc.read_h5ad(adata_path)

# Connect to ChEMBL SQLite
conn = sqlite3.connect(chembl_db)

# 1️⃣ Get molregno for chembl_id
mol_df = pd.read_sql_query("SELECT chembl_id, molregno FROM molecule_dictionary", conn)

# 2️⃣ Get MoA and action_type
moa_df = pd.read_sql_query("SELECT molregno, mechanism_of_action, action_type FROM drug_mechanism", conn)

conn.close()

# 3️⃣ Merge chembl_id → molregno → MoA
chembl_map = mol_df.merge(moa_df, on="molregno", how="left")

# 4️⃣ Collapse multiple MoAs per chembl_id
chembl_map_grouped = chembl_map.groupby('chembl_id').agg({
    'mechanism_of_action': lambda x: '; '.join(sorted(set(x.dropna()))),
    'action_type': lambda x: '; '.join(sorted(set(x.dropna())))
}).reset_index()

# 5️⃣ Merge with AnnData.obs
adata.obs = adata.obs.merge(
    chembl_map_grouped,
    on='chembl_id',
    how='left'
)

# 6️⃣ Quick check
print("Rows with MoA:", adata.obs['mechanism_of_action'].notna().sum())
print("Unique MoA strings:", adata.obs['mechanism_of_action'].nunique())



In [ ]:
# Replace empty strings or blanks with 'Unclear'
adata.obs['mechanism_of_action'] = adata.obs['mechanism_of_action'].replace('', 'Unclear')
adata.obs['mechanism_of_action'] = adata.obs['mechanism_of_action'].fillna('Unclear')

# Now check counts
moa_counts = adata.obs['mechanism_of_action'].value_counts()
print(moa_counts.head(10))  # top 10 MoAs including 'Unclear'

# Count of Unclear rows
num_unclear = (adata.obs['mechanism_of_action'] == 'Unclear').sum()
total_rows = adata.n_obs
print(f"Rows with unclear MoA: {num_unclear} / {total_rows} ({num_unclear/total_rows*100:.2f}%)")

In [ ]:
import numpy as np

# Replace empty strings or whitespace with NaN
adata.obs['action_type'] = adata.obs['action_type'].replace(r'^\s*$', np.nan, regex=True)

# Check result
print("Missing action_type:", adata.obs['action_type'].isna().sum())

In [ ]:
import numpy as np

# Replace empty string with NaN
adata.obs['action_type'] = adata.obs['action_type'].replace('', np.nan)

# Check again
print("Unique action types:", adata.obs['action_type'].nunique(dropna=True))
print("Number of NaN:", adata.obs['action_type'].isna().sum())

In [ ]:
import numpy as np

# Define condition for controls
control_mask = (adata.obs['pert_type'] == 'ctl_vehicle') & \
               (adata.obs['pert_dose'] == 0.1) & \
               (adata.obs['pert_dose_unit'] == '%')

# Replace dose and unit with -666
adata.obs.loc[control_mask, 'pert_dose'] = -666
adata.obs.loc[control_mask, 'pert_dose_unit'] = '-666'

# Check how many were replaced
print("Number of control rows updated:", control_mask.sum())
print(adata.obs.loc[control_mask, ['pert_dose', 'pert_dose_unit']].head())

In [ ]:
import anndata
import pandas as pd

# Load dataset
adata = anndata.read_h5ad("/sybig/home/shs/projects/project1_L1000/datasets/lincs/merge/Level3_twophase_with_chembl_moa_clean.h5ad")

# Make DataFrame of action_type column
df = adata.obs[['action_type']].copy()

# Ensure all entries are strings
df['action_type'] = df['action_type'].astype(str)

# Step 1: Keep only single-action rows (no ';') and ignore 'nan'
single_rows = df[~df['action_type'].str.contains(';') & (df['action_type'] != 'nan')]

# Step 2: Count occurrences of each single action type
single_counts = single_rows['action_type'].value_counts()

# Step 3: Calculate percentage relative to all single-action rows
percent_single = (single_counts / single_counts.sum()) * 100

# Sort descending
percent_single = percent_single.sort_values(ascending=False)

print("Percentage of single-action types:")
print(percent_single)

In [ ]:
import anndata
import pandas as pd

# Load your dataset
adata = anndata.read_h5ad("/sybig/home/shs/projects/project1_L1000/datasets/lincs/merge/Level3_twophase_with_chembl_moa_clean.h5ad")

# Copy action_type column
df = adata.obs[['action_type']].copy()

# Ensure all entries are strings
df['action_type'] = df['action_type'].astype(str)

# Step 1: Calculate single-action percentages
single_rows = df[~df['action_type'].str.contains(';') & (df['action_type'] != 'nan')]
single_counts = single_rows['action_type'].value_counts()
percent_single = (single_counts / single_counts.sum()) * 100

# Step 2: Function to keep the highest single-percentage type in multi-action rows
def keep_highest_single(row):
    if ';' not in row or row == 'nan':
        return row  # keep single-action or NaN
    types = [t.strip() for t in row.split(';')]
    # Pick the type with highest single % (default 0 if missing)
    best_type = max(types, key=lambda t: percent_single.get(t, 0))
    return best_type

# Step 3: Apply function to the column
df['action_type_filtered'] = df['action_type'].apply(keep_highest_single)

# Step 4: Update your adata.obs (optional)
adata.obs['action_type_filtered'] = df['action_type_filtered']

# Check a few examples
print(df.head(10))

In [ ]:
# Function to keep MoA if it contains the action_type substring
def filter_moa_substring(row):
    moa = row['mechanism_of_action']
    action = row['action_type_filtered']
    
    if pd.isna(moa) or action == 'nan' or pd.isna(action):
        return moa  # keep original if NaN
    
    # Split multi-mechanisms if separated by ';'
    moa_list = [m.strip() for m in moa.split(';')]
    
    # Keep mechanisms containing the action_type substring (case-insensitive)
    filtered = [m for m in moa_list if action.upper() in m.upper()]
    
    return '; '.join(filtered) if filtered else pd.NA

# Apply to the dataset
adata.obs['mechanism_of_action_filtered'] = adata.obs.apply(filter_moa_substring, axis=1)

# Check a few examples
print(adata.obs[['action_type_filtered', 'mechanism_of_action', 'mechanism_of_action_filtered']].sample(10, random_state=42))

split_ComBat

In [ ]:
# Step 1 — Load Dataset and Define Biological Context

import scanpy as sc
import pandas as pd

# Load dataset
adata = sc.read_h5ad("/sybig/home/shs/projects/project1_L1000/datasets/lincs/merge/second_phase_combine/final_file_for_split_updated.h5ad")

# Work on obs dataframe
df = adata.obs.copy()
# Fix pert_dose for controls
# Replace -666 or any negative control values with 0
df.loc[df['pert_id'] == 'DMSO', 'pert_dose'] = 0

# Optional: Also ensure no negative doses for other compounds if needed
df.loc[df['pert_dose'] < 0, 'pert_dose'] = 0  # or handle as appropriate

# Define context_key
df["context_key"] = (
    df["cell_id"].astype(str) + "_" +
    df["mechanism_of_action_filtered"].astype(str) + "_" +
    df["pert_dose"].astype(str)
)

In [ ]:
# Remove controls before context counting
non_controls = df[df["pert_id"] != "DMSO"]

# Count unique drugs per context
context_drug_counts = non_controls.groupby("context_key")["pert_id"].nunique()

# Orphan contexts
orphan_contexts = context_drug_counts[context_drug_counts == 1].index
orphans = non_controls[non_controls["context_key"].isin(orphan_contexts)]

# Shared contexts
shared_contexts = context_drug_counts[context_drug_counts >= 2].index
shared = non_controls[non_controls["context_key"].isin(shared_contexts)]

# Controls separately
controls = df[df["pert_id"] == "DMSO"]

In [ ]:
# Step 2 — Check the groups

print("Total rows:", len(df))
print()

print("Controls (DMSO):", len(controls))
print("Unique control contexts:", controls["context_key"].nunique())
print()

print("Orphan rows:", len(orphans))
print("Orphan contexts:", len(orphan_contexts))
print()

print("Shared rows:", len(shared))
print("Shared contexts:", len(shared_contexts))
print()

# Sanity checks
print("Overlap orphan/shared:", set(orphan_contexts).intersection(set(shared_contexts)))
print("Controls inside shared:", controls["context_key"].isin(shared_contexts).sum())
print("Controls inside orphan:", controls["context_key"].isin(orphan_contexts).sum())

In [ ]:
# Step 3 — 5-fold split on shared contexts

from sklearn.model_selection import StratifiedGroupKFold

sgkf = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

splits = []

for train_idx, test_idx in sgkf.split(
        shared,
        y=shared["context_key"],
        groups=shared["pert_id"]):

    shared_train = shared.iloc[train_idx]
    shared_test = shared.iloc[test_idx]

    splits.append((shared_train, shared_test))

In [ ]:
for i, (train, test) in enumerate(splits):

    train_drugs = set(train["pert_id"])
    test_drugs = set(test["pert_id"])

    overlap = train_drugs.intersection(test_drugs)

    print(f"Fold {i}")
    print("train rows:", len(train))
    print("test rows:", len(test))
    print("drug overlap:", len(overlap))
    print()

In [ ]:
# Step 4 — Combine shared folds with orphans + controls

fold_datasets = []

for shared_train, shared_test in splits:

    # Final train = shared_train + orphans + controls
    final_train = pd.concat([shared_train, orphans, controls])

    # Final test = shared_test only
    final_test = shared_test.copy()

    fold_datasets.append((final_train, final_test))

In [ ]:
# Step 5 — Construct Final Train/Test Sets (strict drug-blind)
fold_datasets = []

for shared_train, shared_test in splits:

    # Exclude orphan rows whose drug is in the current test fold
    orphan_train = orphans[~orphans["pert_id"].isin(shared_test["pert_id"])]

    # Combine for final training set
    final_train = pd.concat([shared_train, orphan_train, controls])

    # Final test set = shared_test only
    final_test = shared_test.copy()

    fold_datasets.append((final_train, final_test))

In [ ]:
# Comprehensive context report for all folds
report = []

for i, (train_df, test_df) in enumerate(fold_datasets):
    
    # Unique contexts
    train_contexts = set(train_df["context_key"])
    test_contexts = set(test_df["context_key"])
    
    # Shared contexts
    shared_in_train = train_df[~train_df["context_key"].isin(orphans["context_key"]) & 
                                ~train_df["context_key"].isin(controls["context_key"])]["context_key"].nunique()
    shared_in_test = test_df["context_key"].nunique()
    
    # Orphan contexts in train and test
    orphan_in_train = train_df[train_df["context_key"].isin(orphans["context_key"])]["context_key"].nunique()
    orphan_in_test = test_df[test_df["context_key"].isin(orphans["context_key"])]["context_key"].nunique()
    
    # Control contexts in train and test
    controls_in_train = train_df[train_df["context_key"].isin(controls["context_key"])]["context_key"].nunique()
    controls_in_test = test_df[test_df["context_key"].isin(controls["context_key"])]["context_key"].nunique()
    
    report.append({
        "Fold": i,
        "TRAIN rows": len(train_df),
        "TEST rows": len(test_df),
        "Shared contexts in TRAIN": shared_in_train,
        "Shared contexts in TEST": shared_in_test,
        "Orphan contexts in TRAIN": orphan_in_train,
        "Orphan contexts in TEST": orphan_in_test,
        "Control contexts in TRAIN": controls_in_train,
        "Control contexts in TEST": controls_in_test
    })

# Convert to DataFrame for easy viewing
report_df = pd.DataFrame(report)
report_df

In [ ]:
# Prepare context counts per fold
fold_context_counts = []

for i, (train_df, test_df) in enumerate(fold_datasets):
    train_context_count = train_df["context_key"].nunique()
    test_context_count  = test_df["context_key"].nunique()
    
    fold_context_counts.append({
        "Fold": i,
        "TRAIN context count": train_context_count,
        "TEST context count": test_context_count
    })

# Convert to DataFrame for easy viewing
import pandas as pd
context_counts_df = pd.DataFrame(fold_context_counts)
context_counts_df

In [ ]:
#Step 1 — Convert Your Split DataFrames Back to AnnData
fold_adatas = []

for train_df, test_df in fold_datasets:

    train_idx = adata.obs.index.isin(train_df.index)
    test_idx = adata.obs.index.isin(test_df.index)

    adata_train = adata[train_idx].copy()
    adata_test = adata[test_idx].copy()

    fold_adatas.append((adata_train, adata_test))

In [ ]:
#Step 2 — Run ComBat for Each Fold
import scanpy as sc

batch_key = "pert_itime"

corrected_folds = []

for i, (adata_train, adata_test) in enumerate(fold_adatas):

    print(f"Running ComBat for fold {i}")

    # Combine train and test for consistent correction
    combined = adata_train.concatenate(
        adata_test,
        batch_key="split",
        batch_categories=["train", "test"]
    )

    # Apply ComBat
    sc.pp.combat(
        combined,
        key=batch_key
    )

    # Split back
    train_corrected = combined[combined.obs["split"] == "train"].copy()
    test_corrected = combined[combined.obs["split"] == "test"].copy()

    corrected_folds.append((train_corrected, test_corrected))

In [ ]:
# Batch Effect Correction Using ComBat on Combined Dataset and Time (Cross-Validation Folds)
import scanpy as sc

corrected_folds = []

for i, (adata_train, adata_test) in enumerate(fold_adatas):

    print(f"Running ComBat for fold {i}")

    # Combine train + test
    combined = adata_train.concatenate(
        adata_test,
        batch_key="split",
        batch_categories=["train", "test"]
    )

    # -----------------------------
    # Create combined batch
    # -----------------------------
    combined.obs["combined_batch"] = (
        combined.obs["dataset_name"].astype(str) + "_" +
        combined.obs["pert_itime"].astype(str)
    )

    # Convert to category (important!)
    combined.obs["combined_batch"] = combined.obs["combined_batch"].astype('category')

    # -----------------------------
    # Apply ComBat
    # -----------------------------
    sc.pp.combat(
        combined,
        key="combined_batch"
    )

    # -----------------------------
    # Split back
    # -----------------------------
    train_corrected = combined[combined.obs["split"] == "train"].copy()
    test_corrected = combined[combined.obs["split"] == "test"].copy()

    corrected_folds.append((train_corrected, test_corrected))

In [ ]:
for i, (train, test) in enumerate(corrected_folds):

    train.write(f"fold_{i}_train_combat_both.h5ad")
    test.write(f"fold_{i}_test_combat_both.h5ad")

In [ ]:
import scanpy as sc

# Load your .h5ad dataset
adata = sc.read("/sybig/home/shs/projects/project1_L1000/datasets/lincs/merge/second_phase_combine/final_file_for_split_updated.h5ad")

# Display the basic information about the dataset
print(adata)

# Show the shape of the dataset (number of cells and genes)
print(f"Shape of the dataset: {adata.shape}")

# Check the names of available attributes in the data (e.g., raw data, metadata)
print("Available data attributes:")
print(adata.obs)  # Cell metadata
print(adata.var)  # Gene metadata
print(adata.X)    # Expression matrix

# Optionally, show the first few rows of cell metadata
print("First few rows of cell metadata:")
print(adata.obs.head())

befor_combat

import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Define the path to your AnnData file
# PLEASE UPDATE THIS PATH
adata_path_before = '/sybig/home/shs/projects/project1_L1000/datasets/lincs/merge/second_phase_combine/final_file_for_split_updated.h5ad' 

try:
    adata = sc.read_h5ad(adata_path_before)
    print("AnnData object loaded successfully.")
except FileNotFoundError:
    print(f"ERROR: The file was not found at '{adata_path_before}'.")
    print("Please update the 'adata_path_before' variable to the correct location of your h5ad file.")
    # In a real notebook, you might stop execution here.
    # We will create a dummy object to allow the script to continue for demonstration.
    #adata = sc.AnnData(np.random.rand(100, 200), obs={'cell_id': ['A549']*50 + ['MCF7']*50, 'dataset_name': ['GSE70138']*50 + ['GSE92742']*50, 'pert_type': ['trt_cp']*50 + ['ctl_vehicle']*50})


print("--- Dataset Overview ---")
print(f"Dimensions: {adata.n_obs} observations (samples) × {adata.n_vars} variables (genes)")
print("--- Observation Metadata (.obs) ---")
print("Available columns:", adata.obs.columns.tolist())
print(adata.obs.head())


In [ ]:
#Step 1 — Understand Metadata Composition
print("Total samples:", adata.n_obs)
print("\nCell lines:")
print(adata.obs['cell_id'].value_counts())

print("\nPerturbation types:")
print(adata.obs['pert_type'].value_counts())

print("\nDatasets:")
print(adata.obs['dataset_name'].value_counts())

print("\nPerturbation Time:")
print(adata.obs['pert_itime'].value_counts())

In [ ]:
# -----------------------------
# Step 1: Define subset size
# -----------------------------
n_samples_for_viz = 100000
n_samples_for_viz = min(n_samples_for_viz, adata.n_obs)

print(f"Creating a subset of {n_samples_for_viz} samples...")

# -----------------------------
# Step 2: Stratified Subsetting by 'cell_id'
# -----------------------------

# Get integer positions of each cell_id
cell_counts = adata.obs['cell_id'].value_counts()
cell_proportions = cell_counts / cell_counts.sum()

# Determine number of samples per cell line
samples_per_cell = (cell_proportions * n_samples_for_viz).astype(int)

# Adjust to sum exactly to n_samples_for_viz
diff = n_samples_for_viz - samples_per_cell.sum()
if diff > 0:
    samples_per_cell.iloc[0] += diff

# Collect integer positions
subset_indices = []
for cell_line, n in samples_per_cell.items():
    # Get boolean mask for this cell line
    mask = adata.obs['cell_id'] == cell_line
    # Get integer positions (np.where returns positions)
    positions = np.where(mask)[0]
    # Randomly sample positions
    chosen = np.random.choice(positions, size=n, replace=False)
    subset_indices.extend(chosen)

# Create subset using integer positions
adata_subset_before = adata[subset_indices, :].copy()

print("Subset created.")
print(f"Subset size: {adata_subset_before.n_obs}")

# -----------------------------
# Step 3: Check distributions
# -----------------------------
print("\nCell line distribution in subset (top 20):")
print(adata_subset_before.obs['cell_id'].value_counts().head(20))

print("\nPerturbation types in subset:")
print(adata_subset_before.obs['pert_type'].value_counts())

print("\nPerturbation times in subset (top 20):")
if 'pert_itime' in adata_subset_before.obs.columns:
    print(adata_subset_before.obs['pert_itime'].value_counts().head(20))

In [ ]:
###########################################################
# Ensure PCA exists for subset befor batch correction
###########################################################

if 'pca' not in adata_subset_before.uns:
    print("PCA not found. Running PCA now...")
    sc.tl.pca(adata_subset_before, n_comps=50, svd_solver='arpack')

###########################################################
# Plot explained variance (PERCENTAGE + cumulative)
###########################################################

import numpy as np
import matplotlib.pyplot as plt

print("\n--- Plotting PCA Variance Ratio (Percentage) for subset befor batch correction ---")

# Extract variance ratio safely
var_ratio = adata_subset_before.uns['pca']['variance_ratio']
n_pcs = min(50, len(var_ratio))

# Convert to percentage
var_ratio_percent = var_ratio[:n_pcs] * 100

# Compute cumulative variance
cumulative = np.cumsum(var_ratio_percent)



# Plot
plt.figure(figsize=(10, 6), dpi=150)

plt.plot(range(1, n_pcs + 1), var_ratio_percent, marker='o', label='Individual')
plt.plot(range(1, n_pcs + 1), cumulative, linestyle='--', label='Cumulative')




plt.xlabel('Principal Component')
plt.ylabel('Explained Variance (%)')
plt.title('PCA Explained Variance befor Batch Correction')

plt.ylim(0, 100)
plt.xticks(range(1, n_pcs + 1, 5))
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()

plt.savefig("PCA_variance_ratio_percent_befor.png", dpi=300, bbox_inches='tight')
plt.close()

print("PCA variance (percentage + cumulative) plot befor batch correction saved.")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc

# Step 3: Data Scaling
print("--- Step 3: Scaling Data ---")

# Scale data for PCA
sc.pp.scale(adata_subset_before, max_value=10)


# Step 4: PCA
print("\n--- Step 4: Running PCA ---")

# Run PCA
sc.tl.pca(adata_subset_before, n_comps=50, svd_solver='arpack')

print("PCA calculation complete.")


###########################################################
# Plot explained variance
###########################################################

sc.pl.pca_variance_ratio(
    adata_subset_before,
    log=True,
    n_pcs=50,
    show=False
)

plt.savefig(
    "PCA_variance_ratio.png",
    dpi=300,
    bbox_inches='tight'
)

plt.close()


###########################################################
# PCA by Dataset
###########################################################

if 'dataset_name' in adata_subset_before.obs.columns:

    sc.pl.pca(
        adata_subset_before,
        color='dataset_name',
        title='PCA by Dataset',
        show=False,
        size=15,
        alpha=0.9
    )

    plt.title('PCA by Dataset', fontsize=16, weight='bold')
    plt.gca().tick_params(axis='both', labelsize=10)
    plt.gcf().set_size_inches(12, 10)

    plt.legend(
        loc='center left',
        bbox_to_anchor=(1, 0.5),
        fontsize=12
    )

    plt.savefig(
        "PCA_by_Dataset.png",
        dpi=300,
        bbox_inches='tight'
    )

    plt.close()


###########################################################
# PCA by Top 20 Cell IDs ONLY
###########################################################

if 'cell_id' in adata_subset_before.obs.columns:

    print("\nSelecting Top 20 cell_id values...")

    top20_cellids = (
        adata_subset_before.obs['cell_id']
        .value_counts()
        .head(20)
        .index
    )

    # Work on copy (safe)
    adata_cell20 = adata_subset_before[
        adata_subset_before.obs['cell_id'].isin(top20_cellids)
    ].copy()

    print("Plotting PCA for Top 20 Cell IDs...")

    sc.pl.pca(
        adata_cell20,
        color='cell_id',
        title='PCA by Top 20 Cell Lines',
        show=False,
        size=15,
        alpha=0.9,
        palette=sns.color_palette("tab20", 20)
    )

    plt.title(
        'PCA by Top 20 Cell Lines',
        fontsize=16,
        weight='bold'
    )

    plt.gca().tick_params(axis='both', labelsize=10)

    plt.gcf().set_size_inches(12, 10)

    plt.legend(
        loc='center left',
        bbox_to_anchor=(1, 0.5),
        fontsize=10
    )

    plt.savefig(
        "PCA_by_Top20_Cell_Line.png",
        dpi=300,
        bbox_inches='tight'
    )

    plt.close()


print("\nPCA plots saved successfully.")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.lines as mlines
import scanpy as sc
import pandas as pd

# Section 5: UMAP for Non-Linear Dimensionality Reduction

print("\n--- Section 5: UMAP Embedding with Clean Solid Dots ---")

############################################################
# Step 5a: Compute neighbor graph
############################################################

print("Calculating neighbor graph...")
sc.pp.neighbors(adata_subset_before, n_neighbors=15, n_pcs=50)
print("Neighbor graph complete.\n")


############################################################
# Step 5b: Run UMAP
############################################################

print("Running UMAP...")
sc.tl.umap(adata_subset_before)
print("UMAP calculation complete.\n")


############################################################
# Step 5c: Visualize UMAP
############################################################

# Solid dots instead of faint blur
alpha_val = 0.9

categorical_cols = ['dataset_name', 'cell_id', 'pert_type', 'pert_itime']


for col in categorical_cols:

    if col in adata_subset_before.obs.columns:

        print(f"\nProcessing column: {col}")

        # Work on copy (prevents shrinking dataset repeatedly)
        adata_plot = adata_subset_before.copy()

        n_cats = adata_plot.obs[col].nunique()

        ####################################################
        # Select top 20 categories where needed
        ####################################################

        if col in ['cell_id', 'pert_type', 'pert_itime']:

            top_20_values = (
                adata_plot.obs[col]
                .value_counts()
                .head(20)
                .index
            )

            adata_plot = adata_plot[
                adata_plot.obs[col].isin(top_20_values)
            ]

            print(f"Top 20 values selected for '{col}'.")


        ####################################################
        # Color palette
        ####################################################

        palette = (
            sns.color_palette("tab20", 20)
            if n_cats > 10
            else sns.color_palette("Set2", 10)
        )


        ####################################################
        # Plot UMAP (Example Style)
        ####################################################

        print(f"Plotting UMAP colored by '{col}'...")

        sc.pl.umap(
            adata_plot,
            color=col,
            title=f'UMAP by {col}',
            alpha=alpha_val,        # Solid clusters
            palette=palette,
            legend_loc='right',
            legend_fontsize=8,
            size=15,                # Small clean dots
            frameon=True,
            linewidth=0,            # Sharp circles
            show=False
        )


        ####################################################
        # Styling
        ####################################################

        plt.title(
            f'UMAP by {col}',
            fontsize=16,
            weight='bold'
        )

        plt.gca().tick_params(
            axis='both',
            labelsize=10
        )

        plt.gcf().set_size_inches(12, 10)


        ####################################################
        # Clean legend placement
        ####################################################

        plt.legend(
            loc='center left',
            bbox_to_anchor=(1, 0.5),
            fontsize=10,
            frameon=True
        )


        ####################################################
        # Save Figure
        ####################################################

        plt.savefig(
            f"UMAP_by_{col}.png",
            dpi=300,
            bbox_inches='tight'
        )

        plt.close()


print("\nAll UMAP plots saved successfully.")

after_ComBat

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os   

# -----------------------------
# Step 0: Define path to ComBat folds
# -----------------------------
combat_path = '/sybig/home/shs/projects/project1_L1000/datasets/lincs/merge/split/splits_with_combat_both'

# List all files ending with "_train_combat.h5ad"
train_files = sorted([f for f in os.listdir(combat_path) if f.endswith('_train_combat_both.h5ad')])
test_files  = sorted([f for f in os.listdir(combat_path) if f.endswith('_test_combat_both.h5ad')])

# Pick fold 0 for visualization
train_file = os.path.join(combat_path, train_files[0])
test_file  = os.path.join(combat_path, test_files[0])

print(f"Using train: {train_file}")
print(f"Using test : {test_file}")

# -----------------------------
# Step 1: Load the fold
# -----------------------------
train = sc.read_h5ad(train_file)
test = sc.read_h5ad(test_file)

# Combine train + test
adata_viz = train.concatenate(
    test,
    batch_key="split",
    batch_categories=["train", "test"]
)

print(f"Combined dataset: {adata_viz.n_obs} cells x {adata_viz.n_vars} genes")

In [ ]:
# -----------------------------
# Step 2: Subset for visualization (optional)
# -----------------------------
n_samples_for_viz = 100000
n_samples_for_viz = min(n_samples_for_viz, adata_viz.n_obs)
print(f"Creating a subset of {n_samples_for_viz} samples...")

# Set random seed for reproducibility
np.random.seed(42)

# -----------------------------
# Step 2a: Stratified subsetting by 'cell_id'
# -----------------------------
cell_counts = adata_viz.obs['cell_id'].value_counts()
cell_proportions = cell_counts / cell_counts.sum()

# Determine how many cells per cell line
samples_per_cell = (cell_proportions * n_samples_for_viz).astype(int)

# Adjust to make the sum exactly n_samples_for_viz
diff = n_samples_for_viz - samples_per_cell.sum()
if diff > 0:
    samples_per_cell.iloc[0] += diff

subset_indices = []
for cell_line, n in samples_per_cell.items():
    mask = adata_viz.obs['cell_id'] == cell_line
    positions = np.where(mask)[0]
    # Ensure we don't sample more than available
    chosen = np.random.choice(positions, size=min(n, len(positions)), replace=False)
    subset_indices.extend(chosen)

# Create the subset
adata_subset_after = adata_viz[subset_indices, :].copy()

print("Subset created successfully.")
print(f"Subset size: {adata_subset_after.n_obs} cells")

# -----------------------------
# Step 3: Check distributions
# -----------------------------
print("\nCell line distribution in subset (top 20):")
print(adata_subset_after.obs['cell_id'].value_counts().head(20))

print("\nPerturbation types in subset:")
print(adata_subset_after.obs['pert_type'].value_counts())

if 'pert_itime' in adata_subset_after.obs.columns:
    print("\nPerturbation times in subset (top 20):")
    print(adata_subset_after.obs['pert_itime'].value_counts().head(20))

In [ ]:
###########################################################
# Ensure PCA exists for subset after batch correction
###########################################################

if 'pca' not in adata_subset_after.uns:
    print("PCA not found. Running PCA now...")
    sc.tl.pca(adata_subset_after, n_comps=50, svd_solver='arpack')

###########################################################
# Plot explained variance (PERCENTAGE + cumulative)
###########################################################

import numpy as np
import matplotlib.pyplot as plt

print("\n--- Plotting PCA Variance Ratio (Percentage) for subset after batch correction ---")

# Extract variance ratio safely
var_ratio = adata_subset_after.uns['pca']['variance_ratio']
n_pcs = min(50, len(var_ratio))

# Convert to percentage
var_ratio_percent = var_ratio[:n_pcs] * 100

# Compute cumulative variance
cumulative = np.cumsum(var_ratio_percent)



# Plot
plt.figure(figsize=(10, 6), dpi=150)

plt.plot(range(1, n_pcs + 1), var_ratio_percent, marker='o', label='Individual')
plt.plot(range(1, n_pcs + 1), cumulative, linestyle='--', label='Cumulative')

plt.xlabel('Principal Component')
plt.ylabel('Explained Variance (%)')
plt.title('PCA Explained Variance after Batch Correction')

plt.ylim(0, 100)
plt.xticks(range(1, n_pcs + 1, 5))
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()

plt.savefig("PCA_variance_ratio_percent_after.png", dpi=300, bbox_inches='tight')
plt.close()

print("PCA variance (percentage + cumulative) plot AFTER batch correction saved.")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc

# -----------------------------
# Step 3: Scale data
# -----------------------------
print("--- Step 3: Scaling Data ---")
sc.pp.scale(adata_subset_after, max_value=10)
print("Data scaling complete.")

# -----------------------------
# Step 4: PCA
# -----------------------------
print("\n--- Step 4: Running PCA ---")
sc.tl.pca(adata_subset_after, n_comps=50, svd_solver='arpack')
print("PCA calculation complete.")

# -----------------------------
# Step 4a: Explained variance
# -----------------------------
sc.pl.pca_variance_ratio(
    adata_subset_after,
    log=True,
    n_pcs=50,
    show=False
)
plt.savefig("PCA_variance_ratio.png", dpi=300, bbox_inches='tight')
plt.close()
print("PCA variance ratio plot saved.")

# -----------------------------
# Step 4b: PCA for categorical columns
# -----------------------------
categorical_cols = ['dataset_name', 'cell_id', 'pert_type', 'pert_itime']

for col in categorical_cols:
    if col in adata_subset_after.obs.columns:
        adata_plot = adata_subset_after.copy()
        n_cats = adata_plot.obs[col].nunique()

        # For cell_id, pert_type, pert_itime → take top 20 categories for clarity
        if col in ['cell_id', 'pert_type', 'pert_itime']:
            top20_values = adata_plot.obs[col].value_counts().head(20).index
            adata_plot = adata_plot[adata_plot.obs[col].isin(top20_values)]

        palette = sns.color_palette("tab20", 20) if n_cats > 10 else sns.color_palette("Set2", 10)

        sc.pl.pca(
            adata_plot,
            color=col,
            title=f'PCA by {col}',
            show=False,
            size=15,
            alpha=0.9,
            palette=palette
        )

        plt.gcf().set_size_inches(12, 10)
        plt.gca().tick_params(axis='both', labelsize=10)
        plt.legend(loc='center left', bbox_to_anchor=(1, 0.5), fontsize=10)
        plt.savefig(f"PCA_by_{col}.png", dpi=300, bbox_inches='tight')
        plt.close()
        print(f"PCA by {col} saved.")

print("\nAll PCA plots (dataset_name, cell_id, pert_type, pert_itime) saved successfully.")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
import numpy as np

# -----------------------------
# Step 5: UMAP Embedding
# -----------------------------
print("\n--- Step 5: UMAP Embedding with Clean Solid Dots ---")

# Step 5a: Compute neighbor graph
print("Calculating neighbor graph...")
sc.pp.neighbors(adata_subset_after, n_neighbors=15, n_pcs=50)
print("Neighbor graph complete.\n")

# Step 5b: Run UMAP
print("Running UMAP...")
sc.tl.umap(adata_subset_after)
print("UMAP calculation complete.\n")

# Step 5c: Visualize UMAP
alpha_val = 0.9
categorical_cols = ['dataset_name', 'cell_id', 'pert_type', 'pert_itime']

for col in categorical_cols:
    if col in adata_subset_after.obs.columns:
        print(f"\nProcessing column: {col}")

        # Make a copy to avoid modifying original subset
        adata_plot = adata_subset_after.copy()
        n_cats = adata_plot.obs[col].nunique()

        # For cell_id, pert_type, pert_itime → take top 20 categories
        if col in ['cell_id', 'pert_type', 'pert_itime']:
            top20_values = adata_plot.obs[col].value_counts().head(20).index
            adata_plot = adata_plot[adata_plot.obs[col].isin(top20_values)]
            print(f"Top 20 values selected for '{col}'.")

        # Define color palette
        palette = sns.color_palette("tab20", 20) if n_cats > 10 else sns.color_palette("Set2", 10)

        # Plot UMAP
        sc.pl.umap(
            adata_plot,
            color=col,
            title=f'UMAP by {col}',
            alpha=alpha_val,
            palette=palette,
            legend_loc='right',
            legend_fontsize=8,
            size=15,
            frameon=True,
            linewidth=0,
            show=False
        )

        # Styling
        plt.title(f'UMAP by {col}', fontsize=16, weight='bold')
        plt.gca().tick_params(axis='both', labelsize=10)
        plt.gcf().set_size_inches(12, 10)
        plt.legend(loc='center left', bbox_to_anchor=(1, 0.5), fontsize=10, frameon=True)

        # Save figure
        plt.savefig(f"UMAP_by_{col}.png", dpi=300, bbox_inches='tight')
        plt.close()
        print(f"UMAP by {col} saved.")

print("\nAll UMAP plots saved successfully.")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# -----------------------------
# Extract first 10 PCs
# -----------------------------
var_before = adata_subset_before.uns['pca']['variance_ratio'][:10] * 100  # %
var_after = adata_subset_after.uns['pca']['variance_ratio'][:10] * 100    # %

pcs = np.arange(1, 11)  # PC1 to PC10
width = 0.35

# -----------------------------
# Plot
# -----------------------------
plt.figure(figsize=(12, 6), dpi=150)

# Bars
bars_before = plt.bar(pcs - width/2, var_before, width=width, label='Before Batch Correction', color='#4da6ff', edgecolor='black')
bars_after = plt.bar(pcs + width/2, var_after, width=width, label='After Batch Correction', color='#ff9933', edgecolor='black')

# Add values on top of bars
for i in range(10):
    plt.text(pcs[i] - width/2, var_before[i] + 0.5, f'{var_before[i]:.1f}%', ha='center', va='bottom', fontsize=10)
    plt.text(pcs[i] + width/2, var_after[i] + 0.5, f'{var_after[i]:.1f}%', ha='center', va='bottom', fontsize=10)

# Labels and title
plt.xlabel('Principal Component', fontsize=13, weight='bold')
plt.ylabel('Explained Variance (%)', fontsize=13, weight='bold')
plt.title('Variance Ratio of First 10 PCs (Before vs After Batch Correction)', fontsize=15, weight='bold')

# Ticks and grid
plt.xticks(pcs, fontsize=12)
plt.ylim(0, max(max(var_before), max(var_after)) * 1.2)
plt.yticks(fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.4)

# Legend
plt.legend(fontsize=12, frameon=True)

plt.tight_layout()
plt.savefig("PC1_10_variance_comparison_clean.png", dpi=300)
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns

# -----------------------------
# Step 1: Identify shared cell lines
# -----------------------------
celllines_dataset1 = adata_subset_before.obs.loc[
    adata_subset_before.obs['dataset_name'] == 'GSE70138', 'cell_id'
].unique()
celllines_dataset2 = adata_subset_before.obs.loc[
    adata_subset_before.obs['dataset_name'] == 'GSE92742', 'cell_id'
].unique()

shared_celllines = np.intersect1d(celllines_dataset1, celllines_dataset2)
print("Shared cell lines:", shared_celllines)

# -----------------------------
# Step 2: Filter ctl_vehicle controls
# -----------------------------
adata_before_ctl = adata_subset_before[
    (adata_subset_before.obs['cell_id'].isin(shared_celllines)) &
    (adata_subset_before.obs['pert_type'] == 'ctl_vehicle')
].copy()

adata_after_ctl = adata_subset_after[
    (adata_subset_after.obs['cell_id'].isin(shared_celllines)) &
    (adata_subset_after.obs['pert_type'] == 'ctl_vehicle')
].copy()

# -----------------------------
# Step 3: Compute mean per cell line
# -----------------------------
def mean_per_cellline(adata):
    means = {}
    for cell in adata.obs['cell_id'].unique():
        subset = adata[adata.obs['cell_id'] == cell]
        # convert sparse matrix if needed
        means[cell] = subset.X.mean(axis=0).A1 if hasattr(subset.X, "A1") else subset.X.mean(axis=0)
    return pd.DataFrame(means)

mean_before = mean_per_cellline(adata_before_ctl)
mean_after  = mean_per_cellline(adata_after_ctl)

# -----------------------------
# Step 4: Compute mean difference per cell line
# -----------------------------
# Take mean across genes
mean_before_avg = mean_before.mean(axis=0)
mean_after_avg  = mean_after.mean(axis=0)
mean_diff = mean_after_avg - mean_before_avg

# Sort and select top 3 most changed controls
top3_ctl = mean_diff.sort_values(ascending=False).head(3).index.tolist()
print("Top 3 ctl_vehicle cell lines by mean diff:", top3_ctl)

# -----------------------------
# Step 5: Prepare DataFrames for plotting UMAP
# -----------------------------
def prepare_df(adata, umap=True):
    # Keep only top3 ctl_vehicle cell lines
    mask = adata.obs['cell_id'].isin(top3_ctl)
    df = adata.obs[mask].copy()
    
    if umap:
        df['x'] = adata.obsm['X_umap'][mask, 0]
        df['y'] = adata.obsm['X_umap'][mask, 1]
    else:
        df['x'] = adata.obsm['X_pca'][mask, 0]
        df['y'] = adata.obsm['X_pca'][mask, 1]
    
    return df

df_before = prepare_df(adata_before_ctl, umap=True)
df_after  = prepare_df(adata_after_ctl, umap=True)

# -----------------------------
# Step 6: Plotting function
# -----------------------------
def scatter_shared(df, title="UMAP Scatter", savefile=None):
    cell_lines = sorted(df['cell_id'].unique())
    datasets = sorted(df['dataset_name'].unique())
    
    markers = ['o','s','^']  # one per cell line
    marker_dict = dict(zip(cell_lines, markers))
    
    palette = sns.color_palette("Set2", n_colors=len(datasets))
    color_dict = dict(zip(datasets, palette))
    
    plt.figure(figsize=(8,6))
    
    for cell_line in cell_lines:
        df_sub = df[df['cell_id']==cell_line]
        for dataset in datasets:
            df_sub2 = df_sub[df_sub['dataset_name']==dataset]
            plt.scatter(
                df_sub2['x'],
                df_sub2['y'],
                s=70,
                alpha=0.8,
                marker=marker_dict[cell_line],
                color=color_dict[dataset],
                edgecolor='k',
                linewidth=0.5,
                label=f'{cell_line} - {dataset}'
            )
    
    plt.xlabel('UMAP1', fontsize=12, weight='bold')
    plt.ylabel('UMAP2', fontsize=12, weight='bold')
    plt.title(title, fontsize=14, weight='bold')
    plt.grid(True, linestyle='--', alpha=0.4)
    
    handles, labels = plt.gca().get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    plt.legend(by_label.values(), by_label.keys(), bbox_to_anchor=(1.05,1), loc='upper left', fontsize=10)
    
    plt.tight_layout()
    if savefile:
        plt.savefig(savefile, dpi=300, bbox_inches='tight')
    plt.show()

# -----------------------------
# Step 7: Plot UMAP before and after batch correction
# -----------------------------
scatter_shared(df_before, title="UMAP Before Batch Correction (ctl_vehicle top3)", savefile="UMAP_before_ctl_top3.png")
scatter_shared(df_after, title="UMAP After Batch Correction (ctl_vehicle top3)", savefile="UMAP_after_ctl_top3.png")

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from scipy.stats import pearsonr
import scanpy as sc

# -----------------------------
# RDKit (optional)
# -----------------------------
try:
    from rdkit import Chem
    from rdkit.Chem import AllChem
    RDKIT_AVAILABLE = True
except ImportError:
    RDKIT_AVAILABLE = False
    print("Warning: RDKit not found. Using random fingerprints.")

# -----------------------------
# Functions
# -----------------------------
def smiles_to_fp(smiles, n_bits=1024):
    if not RDKIT_AVAILABLE:
        return np.random.randint(0, 2, n_bits)
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return np.zeros(n_bits)
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=n_bits)
    return np.array(fp)

def evaluate_predictions(y_true, y_pred, label="Model"):
    pearsons = []
    for i in range(len(y_true)):
        r, _ = pearsonr(y_true[i], y_pred[i])
        if np.isnan(r):
            r = 0.0
        pearsons.append(r)
    avg_pearson = np.mean(pearsons)
    avg_r2 = r2_score(y_true, y_pred, multioutput='uniform_average')
    print(f"[{label}] Avg Pearson: {avg_pearson:.3f} | Avg R2: {avg_r2:.3f}")
    return {"avg_pearson": avg_pearson, "avg_r2": avg_r2}

# -----------------------------
# Config
# -----------------------------
split_dir = "/sybig/home/shs/projects/project1_L1000/datasets/lincs/merge/split/splits_with_combat_both"
n_folds = 5

# Folder to save predictions
pred_dir = "predictions_fold"
os.makedirs(pred_dir, exist_ok=True)

# Store results
results = {"global": [], "cell": [], "ridge_delta": [], "rf_delta": []}

# -----------------------------
# Cross-validation loop
# -----------------------------
for fold in range(n_folds):
    print(f"\n================ Fold {fold} ================")

    # Load data
    adata_train = sc.read_h5ad(os.path.join(split_dir, f"fold_{fold}_train_combat_both.h5ad"))
    adata_test  = sc.read_h5ad(os.path.join(split_dir, f"fold_{fold}_test_combat_both.h5ad"))

    df_train = adata_train.obs.copy()
    df_test  = adata_test.obs.copy()

    Y_train = adata_train.X.toarray() if hasattr(adata_train.X, "toarray") else adata_train.X
    Y_test  = adata_test.X.toarray() if hasattr(adata_test.X, "toarray") else adata_test.X

    # -----------------------------
    # Feature Engineering
    # -----------------------------
    all_smiles = pd.concat([df_train['canonical_smiles'], df_test['canonical_smiles']]).unique()
    fp_map = {s: smiles_to_fp(s) for s in all_smiles}

    X_train_drug = np.stack([fp_map[s] for s in df_train['canonical_smiles']])
    X_test_drug  = np.stack([fp_map[s] for s in df_test['canonical_smiles']])

    cell_encoder = OneHotEncoder(sparse_output=False)
    X_train_cells = cell_encoder.fit_transform(df_train[['cell_id']])
    X_test_cells  = cell_encoder.transform(df_test[['cell_id']])

    dose_scaler = StandardScaler()
    X_train_dose = dose_scaler.fit_transform(df_train[['pert_dose']])
    X_test_dose  = dose_scaler.transform(df_test[['pert_dose']])

    X_train = np.hstack([X_train_drug, X_train_cells, X_train_dose])
    X_test  = np.hstack([X_test_drug, X_test_cells, X_test_dose])
    print(f"Feature shapes: Train={X_train.shape}, Test={X_test.shape}")

    # -----------------------------
    # Global mean baseline
    # -----------------------------
    global_mean = np.mean(Y_train, axis=0)
    y_pred_global = np.tile(global_mean, (len(Y_test), 1))
    results["global"].append(evaluate_predictions(Y_test, y_pred_global, label="Global Mean"))

    # -----------------------------
    # Cell-specific mean baseline
    # -----------------------------
    control_mask = df_train['pert_id'] == 'DMSO'
    cell_means = {}
    for cell in df_train['cell_id'].unique():
        mask = (df_train['cell_id'] == cell) & control_mask
        cell_means[cell] = np.mean(Y_train[mask], axis=0) if mask.any() else global_mean

    y_pred_cell = np.array([cell_means.get(c, global_mean) for c in df_test['cell_id']])
    results["cell"].append(evaluate_predictions(Y_test, y_pred_cell, label="Cell-Specific Mean"))

    # -----------------------------
    # Compute delta for training
    # -----------------------------
    Y_train_delta = np.array([
        Y_train[i] - cell_means[df_train.iloc[i]['cell_id']]
        for i in range(len(Y_train))
    ])

    # -----------------------------
    # Ridge Regression on Delta
    # -----------------------------
    ridge = Ridge(alpha=1.0)
    ridge.fit(X_train, Y_train_delta)
    y_pred_ridge_delta = ridge.predict(X_test)
    y_pred_ridge_full = np.array([
        y_pred_ridge_delta[i] + cell_means[df_test.iloc[i]['cell_id']]
        for i in range(len(Y_test))
    ])
    results["ridge_delta"].append(evaluate_predictions(Y_test, y_pred_ridge_full, label="Ridge Regression (Delta)"))

    # -----------------------------
    # Random Forest on Delta
    # -----------------------------
    rf = RandomForestRegressor(n_estimators=50, max_depth=10, n_jobs=16, random_state=42)
    rf.fit(X_train, Y_train_delta)
    y_pred_rf_delta = rf.predict(X_test)
    y_pred_rf_full = np.array([
        y_pred_rf_delta[i] + cell_means[df_test.iloc[i]['cell_id']]
        for i in range(len(Y_test))
    ])
    results["rf_delta"].append(evaluate_predictions(Y_test, y_pred_rf_full, label="Random Forest (Delta)"))

    # -----------------------------
    # Save predictions for this fold
    # -----------------------------
    np.save(os.path.join(pred_dir, f"y_pred_global_fold{fold}.npy"), y_pred_global)
    np.save(os.path.join(pred_dir, f"y_pred_cell_fold{fold}.npy"), y_pred_cell)
    np.save(os.path.join(pred_dir, f"y_pred_ridge_fold{fold}.npy"), y_pred_ridge_full)
    np.save(os.path.join(pred_dir, f"y_pred_rf_fold{fold}.npy"), y_pred_rf_full)
    np.save(os.path.join(pred_dir, f"Y_test_fold{fold}.npy"), Y_test)
    df_test.to_csv(os.path.join(pred_dir, f"df_test_fold{fold}.csv"), index=False)

    print(f"✅ Fold {fold} completed and predictions saved!")

# -----------------------------
# Final Aggregated Results
# -----------------------------
print("\n================ FINAL RESULTS ================")
for model_name, res_list in results.items():
    avg_pearson = np.mean([r['avg_pearson'] for r in res_list])
    avg_r2 = np.mean([r['avg_r2'] for r in res_list])
    print(f"\n[{model_name.upper()}]")
    print(f"Avg Pearson: {avg_pearson:.3f}")
    print(f"Avg R2: {avg_r2:.3f}")